In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- 1. Tiny dataset ----
from datasets import load_dataset

# load dataset
dataset = load_dataset("roneneldan/TinyStories", split="train")

dataset = dataset.select(range(1000))

# combine into one string
text = " ".join(dataset["text"])

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long).to(device)

# ---- 2. Batching ----
block_size = 64

def get_batch():
    start = random.randint(0, len(data) - block_size - 1)
    x = data[start:start+block_size]
    y = data[start+1:start+block_size+1]
    return x, y

# ---- 3. Model ----
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads

        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)

        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        seq_len = x.shape[0]

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # split into heads
        Q = Q.view(seq_len, self.num_heads, self.head_dim)
        K = K.view(seq_len, self.num_heads, self.head_dim)
        V = V.view(seq_len, self.num_heads, self.head_dim)

        # transpose for attention
        Q = Q.transpose(0,1)  # (heads, seq, dim)
        K = K.transpose(0,1)
        V = V.transpose(0,1)

        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)

        mask = torch.tril(torch.ones(scores.size(-2), scores.size(-1), device=x.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)

        out = weights @ V  # (heads, seq, dim)

        # combine heads
        out = out.transpose(0,1).contiguous().view(seq_len, -1)

        return self.fc_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.attn = MultiHeadAttention(embed_size, num_heads=4)
        self.ln1 = nn.LayerNorm(embed_size)

        self.ff = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size)
        )
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # attention + residual
        x = x + self.ff(self.ln2(x))     # feedforward + residual
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, embed_size=128, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_embed = nn.Embedding(max_len, embed_size)

        self.blocks = nn.Sequential(
    TransformerBlock(embed_size),
    TransformerBlock(embed_size),
    TransformerBlock(embed_size),
    TransformerBlock(embed_size),
)

        self.ln = nn.LayerNorm(embed_size)
        self.head = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        seq_len = x.size(0)
        positions = torch.arange(0, seq_len, device=x.device)

        x = self.embed(x) + self.pos_embed(positions)
        x = self.blocks(x)

        x = self.ln(x)
        logits = self.head(x)
        return logits


model = TinyGPT(vocab_size).to(device)

# ---- 4. Training ----
optimizer = optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss()

for step in range(4000):

    x, y = get_batch()

    logits = model(x)
    loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

# ---- 5. Generate ----
model.eval()

x = torch.tensor([data[0].item()], device=device)

for _ in range(60):
    logits = model(x)

    temperature = 0.9
    #probs = torch.softmax(logits[-1] / temperature, dim=0)

    #next_token = torch.multinomial(probs, 1)
    k = 5

    logits = logits[-1] / temperature

    values, indices = torch.topk(logits, k)
    probs = torch.softmax(values, dim=0)
    next_token = indices[torch.multinomial(probs, 1)]
    x = torch.cat([x, next_token])

print("Generated:", decode(x.tolist()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Step 0, Loss: 4.3677849769592285
Step 50, Loss: 2.830263614654541
Step 100, Loss: 2.383770704269409
Step 150, Loss: 2.7566335201263428
Step 200, Loss: 2.4015159606933594
Step 250, Loss: 2.3403286933898926
Step 300, Loss: 2.4937071800231934
Step 350, Loss: 2.549736976623535
Step 400, Loss: 2.5330758094787598
Step 450, Loss: 2.3527755737304688
Step 500, Loss: 2.530745506286621
Step 550, Loss: 2.2278008460998535
Step 600, Loss: 2.337862491607666
Step 650, Loss: 2.3572916984558105
Step 700, Loss: 2.276268482208252
Step 750, Loss: 2.62147855758667
Step 800, Loss: 2.2384843826293945
Step 850, Loss: 2.3819336891174316
Step 900, Loss: 2.4603567123413086
Step 950, Loss: 2.511685371398926
Step 1000, Loss: 2.2081315517425537
Step 1050, Loss: 2.266176700592041
Step 1100, Loss: 2.4616990089416504
Step 1150, Loss: 2.22251558303833
Step 1200, Loss: 2.3144702911376953
Step 1250, Loss: 2.282832145690918
Step 1300, Loss: 2.2071409225463867
Step 1350, Loss: 2.3779656887054443
Step 1400, Loss: 1.920923233